# 90 — Nodal full-node detection, picking, and shot-gather export

This notebook replaces the earlier detection-driven `90_*` workflow.

The key difference is:

> **Detections define event times. SDS extraction defines shot-gather contents. Picks never decide which stations are saved.**

The workflow is:

1. Discover all position-coded `DP*` stations in the SDS archive. This assumes the 1000 Hz `GP*` nodes have already been downsampled to 500 Hz and rewritten as `DP*` using notebook/script `86_*`.
2. Run network coincidence detection on `DPZ` only, using all available nodes in each named time window.
3. Merge duplicate detections across chunk boundaries.
4. For each detection, extract full 3-component `DPE/DPN/DPZ` shot gathers from SDS.
5. Write full-gather MiniSEED, component SEG-Y, and wiggle/image PNGs.
6. Run first-break autopicks and write pick rows.
7. Store the processing catalog in SQLite, with optional CSV exports for inspection.

Later notebooks can use the SQLite catalog to stack repeated nodal shots, compare nodal versus Geode/streamer gathers at common source positions, and build combined super-gathers.

## 1. Imports

This expects your project library layout to include `nodal_shotgather.py` and `segy_tools` under `../lib`, as in the earlier notebooks.

In [1]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
import json
import sqlite3
import uuid
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import Stream, UTCDateTime

# Local project library
import sys
LIB = Path("../lib").resolve()
if str(LIB) not in sys.path:
    sys.path.append(str(LIB))

from nodal_shotgather import (
    DetectionConfig,
    PickingConfig,
    read_deployment_from_sds,
    preprocess_for_detection,
    detect_network_events,
    preprocess_for_picking,
    pick_baer_aic_on_trace,
    try_ar_pick_short_station,
    consensus_pick_for_station,
)

# SEG-Y / plotting helpers. These should exist in ../lib/segy_tools.
try:
    from segy_tools.gather import gather_arrays_to_stream
    from segy_tools.io import write_segy
    from segy_tools.plotting import plot_wiggle_gather, plot_image_gather
    HAVE_SEGY_TOOLS = True
except Exception as e:
    HAVE_SEGY_TOOLS = False
    print("WARNING: could not import segy_tools helpers. SEG-Y/PNG export may be limited.")
    print(e)

No module named 'segy_tools'


## 2. Configuration

Update paths if needed. The SDS root should be the **position-coded SDS archive after running 86**, so all nodes appear as `DP*` at 500 Hz.

In [2]:
# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
SDS_ROOT = PROJECT_ROOT / "nodal_sds_position_codes"

# New output root. Use a new version when the event windows or metadata rules change.
OUT_ROOT = PROJECT_ROOT / "nodal_fullnode_shotgathers_v4"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CSV_EXPORT_DIR = OUT_ROOT / "catalog_exports"
CSV_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Current metadata workbook. This is used only for deriving/annotating time windows
# and for optional approximate Geode-stack metadata matching. The nodal gathers are
# still generated from SDS detections.
METADATA_WORKBOOK = PROJECT_ROOT / "jochen_field_notes_metadata_tables_with_geode_times.xlsx"
if not METADATA_WORKBOOK.exists():
    METADATA_WORKBOOK = PROJECT_ROOT / "metadata" / "jochen_field_notes_metadata_tables_with_geode_times.xlsx"
if not METADATA_WORKBOOK.exists():
    METADATA_WORKBOOK = Path("/Volumes/tachyon/LBSSP_DATA/metadata/jochen_field_notes_metadata_tables_with_geode_times.xlsx")
if not METADATA_WORKBOOK.exists():
    METADATA_WORKBOOK = None

# -----------------------------------------------------------------------------
# Survey-specific time model
# -----------------------------------------------------------------------------
# Correction convention:
#     actual UTC = recorded Geode/laptop file time + correction_s
#
# For refraction surveys, Glenn checked that the EPIC/Jochen Geode laptop was set
# to UTC but 11.5 s fast, hence correction_s = -11.5.
#
# For Daniel/GeoView streamer surveys, evidence indicates the laptop was set to
# local EDT (UTC-4) and was ~5 min 33 s fast, so correction_s is roughly
# +4 h - 5m33s = +14067 s.  T1 streamer currently has no extracted Geode times
# in the workbook, but this rule is here for later.
#
# Also important: Geode stacked-file time is treated as the FINAL trigger time
# in the stack, so nodal individual blows should occur before that corrected
# final-trigger time.
SURVEY_TIME_MODELS = {
    "T1_1m_refraction": {"correction_s": -11.5, "file_time_meaning": "final_trigger"},
    "T1_2m_refraction": {"correction_s": -11.5, "file_time_meaning": "final_trigger"},
    "T3_1m_refraction": {"correction_s": -11.5, "file_time_meaning": "final_trigger"},
    "T4_1m_refraction": {"correction_s": -11.5, "file_time_meaning": "final_trigger"},
    "T1_Streamer_MASW": {"correction_s": 4*3600 - 5*60 - 33, "file_time_meaning": "final_trigger"},
    "T1A_Streamer_MASW": {"correction_s": 4*3600 - 5*60 - 33, "file_time_meaning": "final_trigger"},
    "Streamer/MASW main transect": {"correction_s": 4*3600 - 5*60 - 33, "file_time_meaning": "final_trigger"},
    "Streamer/MASW western transect": {"correction_s": 4*3600 - 5*60 - 33, "file_time_meaning": "final_trigger"},
}

# Approximate stack windows for matching/detection-window expansion.
DEFAULT_SECONDS_PER_BLOW = 6.0
MIN_STACK_DURATION_S = 30.0
MAX_STACK_DURATION_S = 180.0
DEFAULT_STACK_DURATION_S = 120.0
FINAL_TRIGGER_MARGIN_S = 5.0
TIMEWINDOW_PAD_S = 30.0
METADATA_MATCH_TOLERANCE_S = MAX_STACK_DURATION_S

# -----------------------------------------------------------------------------
# Named processing windows.
# Fallback windows are UTC SmartSolo/SDS times from the earlier 90_* notebook.
# If the current Excel metadata workbook is available, refraction windows are
# updated from Geode file times using the fixed clock correction and final-trigger
# convention. This expands the start time earlier so the first blows in a stack
# are not missed.
# -----------------------------------------------------------------------------
FALLBACK_TIMEWINDOWS = {
    "T1_N1_Streamer": (UTCDateTime("2026-05-16T17:13:00"), UTCDateTime("2026-05-16T21:48:00")),
    "T1_N2_Nodal1": (UTCDateTime("2026-05-17T16:00:00"), UTCDateTime("2026-05-17T19:00:00")),
    "T1_N2_Refraction1m": (UTCDateTime("2026-05-18T16:03:54"), UTCDateTime("2026-05-18T18:38:33")),
    "T1_N2_Refraction2m": (UTCDateTime("2026-05-18T20:18:44"), UTCDateTime("2026-05-18T23:12:53")),
    "T1_N2_Nodal2": (UTCDateTime("2026-05-19T12:59:00"), UTCDateTime("2026-05-19T13:21:00")),
    "T1_N3_Nodal3": (UTCDateTime("2026-05-19T13:59:00"), UTCDateTime("2026-05-19T14:15:00")),
    "T3_N4_Refraction1am": (UTCDateTime("2026-05-19T16:02:00"), UTCDateTime("2026-05-19T18:25:00")),
}

SHEET_TO_TIMEWINDOW = {
    "T1_1m_Refraction": "T1_N2_Refraction1m",
    "T1_2m_Refraction": "T1_N2_Refraction2m",
    "T3_1m_Refraction": "T3_N4_Refraction1am",
}

SHEET_TO_SURVEY = {
    "T1_1m_Refraction": "T1_1m_refraction",
    "T1_2m_Refraction": "T1_2m_refraction",
    "T1_Streamer_MASW": "T1_Streamer_MASW",
    "T1A_Streamer_MASW": "T1A_Streamer_MASW",
    "T3_1m_Refraction": "T3_1m_refraction",
    "T4_1m_Refraction": "T4_1m_refraction",
}

def _pd_time_to_utcdatetime(t) -> UTCDateTime | None:
    if pd.isna(t):
        return None
    return UTCDateTime(pd.Timestamp(t).to_pydatetime())


def stack_duration_from_row(row) -> float:
    """Estimated duration before final trigger covered by a Geode stack."""
    vals = []
    for col in ["n_blows", "n_shots"]:
        if col in row.index:
            try:
                v = float(row.get(col))
                if np.isfinite(v) and v > 1:
                    vals.append(v)
            except Exception:
                pass
    if not vals:
        return DEFAULT_STACK_DURATION_S
    return float(min(MAX_STACK_DURATION_S, max(MIN_STACK_DURATION_S, max(vals) * DEFAULT_SECONDS_PER_BLOW)))


def derive_timewindows_from_metadata(workbook: Path | None, fallback: dict[str, tuple[UTCDateTime, UTCDateTime]]):
    windows = dict(fallback)
    if workbook is None or not Path(workbook).exists():
        print("No current metadata workbook found; using fallback time windows.")
        return windows

    # 1. Use Acquisition_Summary for simple known UTC windows when present.
    try:
        acq = pd.read_excel(workbook, sheet_name="Acquisition_Summary")
        for _, r in acq.iterrows():
            transect = str(r.get("transect", ""))
            activity = str(r.get("activity", "")).lower()
            start = pd.to_datetime(r.get("start_time"), errors="coerce", utc=True)
            end = pd.to_datetime(r.get("end_time"), errors="coerce", utc=True)
            if pd.isna(start) or pd.isna(end):
                continue
            label = None
            if transect == "T1" and "masw" in activity and "streamer" in activity:
                label = "T1_N1_Streamer"
            elif transect == "T1" and "dense hammer" in activity:
                label = "T1_N2_Nodal1"
            elif transect == "T1" and "nodal survey2" in activity:
                label = "T1_N2_Nodal2"
            elif transect == "T1" and "return to survey1" in activity:
                label = "T1_N3_Nodal3"
            elif transect == "T3" and "1 m refraction" in activity:
                label = "T3_N4_Refraction1am"
            if label:
                windows[label] = (UTCDateTime(start.to_pydatetime()), UTCDateTime(end.to_pydatetime()))
    except Exception as e:
        print("Could not read Acquisition_Summary for time windows:", e)

    # 2. Use Geode file times for refraction windows, expanded backwards because
    # Geode file time is the final trigger in the stack.
    for sheet, label in SHEET_TO_TIMEWINDOW.items():
        try:
            df = pd.read_excel(workbook, sheet_name=sheet)
        except Exception:
            continue
        if "geode_laptop_starttime" not in df.columns:
            continue
        t = pd.to_datetime(df["geode_laptop_starttime"], errors="coerce")
        good = df[t.notna()].copy()
        if len(good) == 0:
            continue
        survey_name = SHEET_TO_SURVEY.get(sheet, sheet)
        correction_s = SURVEY_TIME_MODELS.get(survey_name, {}).get("correction_s", 0.0)
        corrected = pd.to_datetime(good["geode_laptop_starttime"], errors="coerce", utc=True) + pd.to_timedelta(correction_s, unit="s")
        durations = good.apply(stack_duration_from_row, axis=1)
        starts = corrected - pd.to_timedelta(durations + TIMEWINDOW_PAD_S, unit="s")
        ends = corrected + pd.to_timedelta(FINAL_TRIGGER_MARGIN_S + TIMEWINDOW_PAD_S, unit="s")
        start = starts.min()
        end = ends.max()
        if pd.notna(start) and pd.notna(end):
            windows[label] = (UTCDateTime(start.to_pydatetime()), UTCDateTime(end.to_pydatetime()))

    return windows

TIMEWINDOWS = derive_timewindows_from_metadata(METADATA_WORKBOOK, FALLBACK_TIMEWINDOWS)

# Leave as None to run all windows, or set a list for testing.
RUN_LABELS = None  # e.g., ["T1_N2_Refraction1m"] for testing

# -----------------------------------------------------------------------------
# Detection and extraction settings
# -----------------------------------------------------------------------------
DETECTION_CHANNEL = "DPZ"       # Z-only detection on all downsampled/real nodes
EXTRACTION_CHANNELS = ["DPE", "DPN", "DPZ"]
DETECTION_SAMPLE_RATE_HZ = 500.0

# Exclude known moving trigger/source node if it appears in SDS.
EXCLUDE_STATIONS = {"12806", "012806", "45012806"}

# Chunked detection avoids loading many hours of data at once.
CHUNK_SECONDS = 60.0
CHUNK_OVERLAP_SECONDS = 2.0

# Event extraction windows for full shot gathers.
# These are relative to detection on_time. Increase pre_s if first breaks are clipped.
GATHER_PRE_S = 0.10
GATHER_POST_S = 0.90

# Avoid detecting/keeping duplicate events on chunk overlaps or repeated trigger branches.
MERGE_TOLERANCE_S = 0.08
MIN_EVENT_SPACING_S = 0.20

# Development limits
MAX_EVENTS_PER_WINDOW = None  # e.g. 10 for testing
WRITE_MSEED = True
WRITE_SEGY = True
WRITE_PNG = True
MAKE_PICK_DIAGNOSTICS = False
EXPORT_CSV_SNAPSHOTS = True

# Detection thresholds. Tune after looking at the event catalog.
det_cfg = DetectionConfig(
    freqmin=5.0,
    freqmax=180.0,
    corners=4,
    zerophase=True,
    sta_seconds=0.02,
    lta_seconds=0.30,
    threshold_on=4.0,
    threshold_off=1.5,
    min_channels=10,   # all 36 nodes should now be available as DPZ; tune this
    min_snr=6.0,
    pretrigger_seconds=GATHER_PRE_S,
    posttrigger_seconds=GATHER_POST_S,
)

pick_cfg = PickingConfig(
    freqmin=10.0,
    freqmax=150.0,
    corners=4,
    zerophase=False,
    pick_tolerance_s=0.02,
    min_votes=2,
    include_ar_s=False,
    mute_seconds=0.02,
)

print("SDS_ROOT:", SDS_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CATALOG_DB:", CATALOG_DB)
print("METADATA_WORKBOOK:", METADATA_WORKBOOK)
print("RUN_LABELS:", RUN_LABELS)
print("Processing windows:")
for k, (a, b) in TIMEWINDOWS.items():
    print(f"  {k}: {a}  ->  {b}  ({b-a:.1f} s)")

# Safety and metadata matching
MIN_FREE_GB = 50.0
# Rerun policy: delete previous nodal rows and products from this output root before processing.
# Set False if you are deliberately appending a second nodal processing run to the same catalog.
REPLACE_EXISTING_NODAL_RUNS = True


SDS_ROOT: /Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4
CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
METADATA_WORKBOOK: /Volumes/tachyon/LBSSP_DATA/metadata/jochen_field_notes_metadata_tables_with_geode_times.xlsx
RUN_LABELS: None
Processing windows:
  T1_N1_Streamer: 2026-05-16T17:13:00.000000Z  ->  2026-05-16T21:48:00.000000Z  (16500.0 s)
  T1_N2_Nodal1: 2026-05-17T16:00:00.000000Z  ->  2026-05-17T19:00:00.000000Z  (10800.0 s)
  T1_N2_Refraction1m: 2026-05-18T16:02:18.500000Z  ->  2026-05-18T18:38:56.500000Z  (9398.0 s)
  T1_N2_Refraction2m: 2026-05-18T20:17:02.500000Z  ->  2026-05-18T23:13:16.500000Z  (10574.0 s)
  T1_N2_Nodal2: 2026-05-19T12:59:00.000000Z  ->  2026-05-19T13:21:00.000000Z  (1320.0 s)
  T1_N3_Nodal3: 2026-05-19T13:59:00.000000Z  ->  2026-05-19T14:15:00.000000Z  (960.0 s)
  T3_N4_Refraction1am: 2026-05-19T16:02:00.000000Z  ->  2026-05-19T18:25:00.000000Z  (8580.0 s)

## 3. SQLite catalog schema

SQLite is the source of truth. CSVs are exported only as snapshots for inspection.

In [3]:

def connect_catalog(db_path: Path) -> sqlite3.Connection:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA busy_timeout = 30000")
    return conn


def table_columns(conn: sqlite3.Connection, table: str) -> set[str]:
    try:
        return {row[1] for row in conn.execute(f"PRAGMA table_info({table})").fetchall()}
    except Exception:
        return set()


def ensure_columns(conn: sqlite3.Connection, table: str, columns: dict[str, str]):
    """Add missing columns to an existing SQLite table.

    This lets the nodal notebook share the same SQLite catalog first populated by
    the Geode metadata notebook, whose tables have a Geode-oriented schema.
    """
    existing = table_columns(conn, table)
    for name, decl in columns.items():
        if name not in existing:
            conn.execute(f"ALTER TABLE {table} ADD COLUMN {name} {decl}")
    conn.commit()


def init_catalog(conn: sqlite3.Connection):
    # Create tables if this notebook is run before 89_*. If 89_* already created
    # the tables, we add the nodal-specific columns below.
    conn.executescript(
    """
    CREATE TABLE IF NOT EXISTS processing_runs (
        run_id TEXT PRIMARY KEY,
        notebook_name TEXT,
        run_time_utc TEXT,
        input_sds_root TEXT,
        output_root TEXT,
        parameters_json TEXT,
        notes TEXT
    );

    CREATE TABLE IF NOT EXISTS receiver_geometry (
        geometry_id TEXT,
        event_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        receiver_index INTEGER,
        receiver_x_m REAL,
        receiver_y_m REAL,
        receiver_spacing_m REAL,
        component TEXT,
        sample_rate_hz REAL,
        geometry_status TEXT,
        geometry_note TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS shot_events (
        event_id TEXT PRIMARY KEY,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        survey_type TEXT,
        shot_no INTEGER,
        file_no INTEGER,
        source_x_m REAL,
        source_type TEXT,
        n_blows INTEGER,
        n_shots INTEGER,
        operator TEXT,
        plate_type TEXT,
        shot_time_local TEXT,
        shot_time_utc TEXT,
        receiver_first_m REAL,
        receiver_last_m REAL,
        receiver_spacing_m REAL,
        nominal_shot_spacing_m REAL,
        geode_read_ok INTEGER,
        geode_read_format TEXT,
        geode_n_traces INTEGER,
        geode_sampling_rate_hz REAL,
        geode_duration_s_first_trace REAL,
        geode_file_path TEXT,
        geode_folder TEXT,
        geode_match_status TEXT,
        geode_match_note TEXT,
        source_page TEXT,
        confidence TEXT,
        review_status TEXT,
        comment TEXT,
        metadata_source_sheet TEXT,
        extra_json TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS shot_gather_files (
        gather_file_id TEXT PRIMARY KEY,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        component TEXT,
        file_type TEXT,
        file_path TEXT,
        source_file_no INTEGER,
        n_traces INTEGER,
        n_receivers INTEGER,
        sample_rate_hz REAL,
        duration_s REAL,
        processing_level TEXT,
        geometry_id TEXT,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS trace_index (
        trace_index_id TEXT PRIMARY KEY,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        line TEXT,
        transect TEXT,
        survey TEXT,
        source_x_m REAL,
        receiver_index INTEGER,
        receiver_x_m REAL,
        offset_m REAL,
        component TEXT,
        file_path TEXT,
        trace_number_in_file INTEGER,
        sample_rate_hz REAL,
        npts INTEGER,
        amplitude_scale REAL,
        run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS picks (
        run_id TEXT,
        event_id TEXT,
        gather_id TEXT,
        instrument_system TEXT,
        network TEXT,
        station TEXT,
        location TEXT,
        channel TEXT,
        component TEXT,
        receiver_x_m REAL,
        source_x_m REAL,
        offset_m REAL,
        pick_time_utc TEXT,
        pick_time_relative_s REAL,
        phase TEXT,
        picker TEXT,
        pick_quality TEXT,
        snr REAL,
        details_json TEXT
    );

    CREATE TABLE IF NOT EXISTS processing_errors (
        run_id TEXT,
        context TEXT,
        label TEXT,
        event_id TEXT,
        error_text TEXT,
        traceback TEXT,
        time_utc TEXT
    );
    """
    )

    ensure_columns(conn, "processing_runs", {
        "input_sds_root": "TEXT", "output_root": "TEXT", "input_metadata_xlsx": "TEXT",
        "input_geode_file_times_csv": "TEXT", "catalog_db": "TEXT", "parameters_json": "TEXT",
        "notes": "TEXT",
    })

    ensure_columns(conn, "receiver_geometry", {
        "network": "TEXT", "location": "TEXT", "station": "TEXT", "elevation_m": "REAL",
        "channel_family": "TEXT", "active_start_utc": "TEXT", "active_end_utc": "TEXT",
        "event_id": "TEXT", "instrument_system": "TEXT", "line": "TEXT", "transect": "TEXT",
        "survey": "TEXT", "receiver_index": "INTEGER", "receiver_x_m": "REAL", "receiver_y_m": "REAL",
        "receiver_spacing_m": "REAL", "component": "TEXT", "sample_rate_hz": "REAL",
        "geometry_status": "TEXT", "geometry_note": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "shot_events", {
        "instrument_system": "TEXT", "line": "TEXT", "network": "TEXT", "location": "TEXT",
        "timewindow_label": "TEXT", "geometry_id": "TEXT", "detection_time_utc": "TEXT",
        "on_time_utc": "TEXT", "off_time_utc": "TEXT", "duration_s": "REAL",
        "source_x_m": "REAL", "source_type": "TEXT", "operator": "TEXT", "plate_type": "TEXT",
        "n_blows": "INTEGER", "n_shots": "INTEGER", "matched_metadata_event_id": "TEXT",
        "metadata_match_status": "TEXT", "metadata_match_time_error_s": "REAL",
        "detection_n_seed_ids": "INTEGER", "detection_n_stations": "INTEGER",
        "detection_seed_ids": "TEXT", "detection_stations": "TEXT", "snr_rms": "REAL",
        "n_receivers_extracted": "INTEGER", "n_traces_extracted": "INTEGER", "status": "TEXT",
        "notes": "TEXT", "survey": "TEXT", "survey_type": "TEXT", "shot_no": "INTEGER", "file_no": "INTEGER",
        "shot_time_utc": "TEXT", "metadata_source_sheet": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "shot_gather_files", {
        "gather_file_id": "TEXT", "event_id": "TEXT", "gather_id": "TEXT", "instrument_system": "TEXT",
        "line": "TEXT", "network": "TEXT", "location": "TEXT", "transect": "TEXT", "survey": "TEXT",
        "timewindow_label": "TEXT", "component": "TEXT", "file_type": "TEXT", "file_path": "TEXT",
        "source_file_no": "INTEGER", "n_traces": "INTEGER", "n_receivers": "INTEGER",
        "sample_rate_hz": "REAL", "duration_s": "REAL", "pre_s": "REAL", "post_s": "REAL",
        "processing_level": "TEXT", "geometry_id": "TEXT", "run_id": "TEXT",
    })

    ensure_columns(conn, "trace_index", {
        "trace_index_id": "TEXT", "event_id": "TEXT", "gather_id": "TEXT", "instrument_system": "TEXT",
        "line": "TEXT", "network": "TEXT", "station": "TEXT", "location": "TEXT", "channel": "TEXT",
        "component": "TEXT", "source_x_m": "REAL", "receiver_index": "INTEGER", "receiver_x_m": "REAL",
        "offset_m": "REAL", "starttime_utc": "TEXT", "file_path": "TEXT", "trace_number_in_file": "INTEGER",
        "trace_index": "INTEGER", "sample_rate_hz": "REAL", "sampling_rate_hz": "REAL", "npts": "INTEGER",
        "amplitude_scale": "REAL", "run_id": "TEXT",
    })

    conn.commit()


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
conn = connect_catalog(CATALOG_DB)
init_catalog(conn)

if REPLACE_EXISTING_NODAL_RUNS:
    # Remove prior nodal catalog rows. Output files are not deleted automatically;
    # using a new OUT_ROOT version is safer for reruns.
    for sql in [
        "DELETE FROM picks WHERE instrument_system = 'nodal'",
        "DELETE FROM trace_index WHERE instrument_system = 'nodal'",
        "DELETE FROM shot_gather_files WHERE instrument_system = 'nodal'",
        "DELETE FROM receiver_geometry WHERE instrument_system = 'nodal'",
        "DELETE FROM shot_events WHERE instrument_system = 'nodal'",
    ]:
        try:
            conn.execute(sql)
        except Exception as e:
            print(f"Nodal cleanup skipped/failed for {sql}: {e}")
    conn.commit()
    print("Removed previous nodal catalog rows before this run.")

params = {
    "SDS_ROOT": str(SDS_ROOT),
    "OUT_ROOT": str(OUT_ROOT),
    "CATALOG_DB": str(CATALOG_DB),
    "TIMEWINDOWS": {k: [str(v[0]), str(v[1])] for k, v in TIMEWINDOWS.items()},
    "RUN_LABELS": RUN_LABELS,
    "DETECTION_CHANNEL": DETECTION_CHANNEL,
    "EXTRACTION_CHANNELS": EXTRACTION_CHANNELS,
    "CHUNK_SECONDS": CHUNK_SECONDS,
    "CHUNK_OVERLAP_SECONDS": CHUNK_OVERLAP_SECONDS,
    "GATHER_PRE_S": GATHER_PRE_S,
    "GATHER_POST_S": GATHER_POST_S,
    "MERGE_TOLERANCE_S": MERGE_TOLERANCE_S,
    "MIN_EVENT_SPACING_S": MIN_EVENT_SPACING_S,
    "MIN_FREE_GB": MIN_FREE_GB,
    "METADATA_MATCH_TOLERANCE_S": METADATA_MATCH_TOLERANCE_S,
    "REPLACE_EXISTING_NODAL_RUNS": REPLACE_EXISTING_NODAL_RUNS,
    "det_cfg": asdict(det_cfg),
    "pick_cfg": asdict(pick_cfg),
}

# Insert only columns that exist in processing_runs, since 89_* and 90_* may
# have slightly different run-provenance schemas.
run_row = {
    "run_id": RUN_ID,
    "notebook_name": "90_nodal_fullnode_detection_pick_and_shotgathers.ipynb",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "input_sds_root": str(SDS_ROOT),
    "output_root": str(OUT_ROOT),
    "catalog_db": str(CATALOG_DB),
    "parameters_json": json.dumps(params, indent=2),
    "notes": "Full-node nodal shot-gather factory; DPZ detection, DP[ENZ] extraction, metadata matching to Geode catalog.",
}
cols = list(table_columns(conn, "processing_runs") & set(run_row.keys()))
pd.DataFrame([{k: run_row[k] for k in cols}]).to_sql("processing_runs", conn, if_exists="append", index=False)
conn.commit()
print("RUN_ID:", RUN_ID)


Removed previous nodal catalog rows before this run.
RUN_ID: 20260617T230552Z_da97ed5d


## 4. Utility functions

Position-coded stations are interpreted as centimetres along the line, e.g. `08808 -> 88.08 m`.

In [4]:
def parse_label(label: str) -> tuple[str, str]:
    """Parse labels like T1_N1_Streamer into network/location."""
    parts = str(label).split("_")
    if len(parts) < 2:
        raise ValueError(f"Cannot parse network/location from label: {label}")
    return parts[0], parts[1]


def station_to_x_m(station: str) -> float:
    """Position-coded station name -> x position in metres."""
    s = str(station).strip()
    if not s.isdigit():
        return np.nan
    return int(s) / 100.0


def channel_to_component(channel: str) -> str:
    return str(channel)[-1].upper()


def stream_stations(st: Stream) -> list[str]:
    return sorted({str(tr.stats.station) for tr in st})


def stream_seed_ids(st: Stream) -> list[str]:
    return sorted({tr.id for tr in st})


def discover_sds_stations(
    sds_root: Path,
    network: str,
    location: str,
    channel_pattern: str = "DPZ",
    exclude_stations: set[str] | None = None,
) -> list[str]:
    """Discover station codes with files matching an SDS channel selector."""
    exclude_stations = exclude_stations or set()
    root = Path(sds_root)
    stations = set()
    # Pattern: YEAR/NET/STA/CHAN.D/NET.STA.LOC.CHAN.D.YEAR.JDAY
    for p in root.rglob(f"{network}.*.{location}.{channel_pattern}.D.*.*"):
        if p.name.startswith("._"):
            continue
        try:
            parts = p.name.split(".")
            if len(parts) >= 7:
                net, sta, loc, cha = parts[:4]
                if net == network and loc == location:
                    if sta not in exclude_stations:
                        stations.add(sta)
        except Exception:
            pass
    return sorted(stations, key=lambda s: (station_to_x_m(s), s))


def build_geometry_df(label: str, start: UTCDateTime, end: UTCDateTime, stations: list[str]) -> pd.DataFrame:
    network, location = parse_label(label)
    rows = []
    for sta in stations:
        rows.append({
            "run_id": RUN_ID,
            "geometry_id": f"{label}_{network}_{location}_DPall",
            "instrument_system": "nodal",
            "line": network,
            "network": network,
            "location": location,
            "station": sta,
            "receiver_x_m": station_to_x_m(sta),
            "receiver_y_m": 0.0,
            "elevation_m": 0.0,
            "channel_family": "DP",
            "sample_rate_hz": DETECTION_SAMPLE_RATE_HZ,
            "active_start_utc": str(start),
            "active_end_utc": str(end),
        })
    return pd.DataFrame(rows)


def utc_to_str(x) -> str | None:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    try:
        return str(UTCDateTime(x))
    except Exception:
        return str(x)


def log_error(conn, context, label=None, event_id=None, exc=None):
    conn.execute(
        "INSERT INTO processing_errors VALUES (?, ?, ?, ?, ?, ?, ?)",
        (
            RUN_ID,
            context,
            label,
            event_id,
            str(exc),
            traceback.format_exc(),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    conn.commit()


def check_free_space(path: Path, min_free_gb: float = MIN_FREE_GB):
    import shutil
    usage = shutil.disk_usage(Path(path))
    free_gb = usage.free / (1024**3)
    if free_gb < min_free_gb:
        raise RuntimeError(f"Only {free_gb:.1f} GiB free at {path}; stopping before writing more outputs.")
    return free_gb


## 5. Detection helpers

Detection is run on `DPZ` only, after the `86_*` downsampling step has converted `GP*` data to `DP*` channels. The resulting detection stream should include both original 500 Hz nodes and downsampled 1000 Hz nodes.

In [5]:
def iter_time_chunks(start: UTCDateTime, end: UTCDateTime, chunk_s: float, overlap_s: float):
    t = UTCDateTime(start)
    i = 0
    while t < end:
        core_start = t
        core_end = min(t + chunk_s, end)
        read_start = max(start, core_start - overlap_s)
        read_end = min(end, core_end + overlap_s)
        yield i, core_start, core_end, read_start, read_end
        t = core_end
        i += 1


def read_detection_stream(label: str, read_start: UTCDateTime, read_end: UTCDateTime, stations: list[str]) -> Stream:
    network, location = parse_label(label)
    # Read station='*' first because EnhancedSDSClient supports wildcards.
    st = read_deployment_from_sds(
        SDS_ROOT,
        network=network,
        location=location,
        station="*",
        channel=DETECTION_CHANNEL,
        starttime=read_start,
        endtime=read_end,
        merge=True,
        verbose=False,
    )
    # Keep only expected/excluded-filtered stations.
    keep = set(stations)
    st = Stream([tr for tr in st if str(tr.stats.station) in keep])
    if len(st):
        st.sort(keys=["station", "channel"])
    return st


def run_detection_for_window(label: str, start: UTCDateTime, end: UTCDateTime, stations: list[str]) -> pd.DataFrame:
    rows = []
    chunk_out = OUT_ROOT / label / "tables" / "chunk_detections"
    chunk_out.mkdir(parents=True, exist_ok=True)

    for chunk_index, core_start, core_end, read_start, read_end in iter_time_chunks(start, end, CHUNK_SECONDS, CHUNK_OVERLAP_SECONDS):
        print(f"{label} chunk {chunk_index}: {core_start} to {core_end}")
        try:
            st_raw = read_detection_stream(label, read_start, read_end, stations)
            if len(st_raw) == 0:
                print("  no traces")
                continue
            print(f"  read {len(st_raw)} traces, {len(stream_stations(st_raw))} stations")

            st_det = preprocess_for_detection(st_raw, det_cfg)
            df = detect_network_events(st_det, det_cfg, write_mseed=False, make_plots=False)
            if len(df) == 0:
                continue

            # Keep detections whose on_time lies inside the core chunk, avoiding duplicate overlap products.
            df = df.copy()
            df["on_time_dt"] = df["on_time"].apply(lambda x: UTCDateTime(x))
            df = df[(df["on_time_dt"] >= core_start) & (df["on_time_dt"] < core_end)].copy()
            if len(df) == 0:
                continue

            df["run_id"] = RUN_ID
            df["timewindow_label"] = label
            network, location = parse_label(label)
            df["network"] = network
            df["location"] = location
            df["chunk_index"] = chunk_index
            df["core_start_utc"] = str(core_start)
            df["core_end_utc"] = str(core_end)
            df["read_start_utc"] = str(read_start)
            df["read_end_utc"] = str(read_end)
            df["n_available_detection_stations"] = len(stations)
            df = df.drop(columns=["on_time_dt"])

            outcsv = chunk_out / f"{label}_{network}_{location}_{chunk_index:06d}_{core_start.strftime('%Y%m%dT%H%M%S')}_{core_end.strftime('%Y%m%dT%H%M%S')}_detections.csv"
            df.to_csv(outcsv, index=False)
            rows.append(df)
            print(f"  detections: {len(df)}")
        except Exception as e:
            print(f"  ERROR: {e}")
            log_error(conn, "detection_chunk", label=label, exc=e)

    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    return out


def merge_detection_catalog(df: pd.DataFrame, tolerance_s: float = 0.08, min_spacing_s: float = 0.20) -> pd.DataFrame:
    """Merge near-duplicate detections, keeping the row with largest n_seed_ids/SNR."""
    if len(df) == 0:
        return df.copy()

    d = df.copy()
    d["_on"] = d["on_time"].apply(lambda x: UTCDateTime(x).timestamp)
    d = d.sort_values("_on").reset_index(drop=True)

    groups = []
    current = [0]
    for i in range(1, len(d)):
        if d.loc[i, "_on"] - d.loc[current[-1], "_on"] <= tolerance_s:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)

    keep_rows = []
    for g in groups:
        sub = d.loc[g].copy()
        # Prefer more triggering channels/stations, then higher SNR if present.
        sort_cols = []
        ascending = []
        for c in ["n_seed_ids", "n_stations", "snr_rms"]:
            if c in sub.columns:
                sort_cols.append(c)
                ascending.append(False)
        if sort_cols:
            sub = sub.sort_values(sort_cols, ascending=ascending)
        keep_rows.append(sub.iloc[0])

    merged = pd.DataFrame(keep_rows).drop(columns=["_on"], errors="ignore").reset_index(drop=True)

    # Enforce minimum spacing by keeping stronger row when detections are still too close.
    if len(merged) <= 1:
        return merged
    merged["_on"] = merged["on_time"].apply(lambda x: UTCDateTime(x).timestamp)
    final = []
    for _, row in merged.sort_values("_on").iterrows():
        if not final:
            final.append(row)
            continue
        if row["_on"] - final[-1]["_on"] < min_spacing_s:
            prev = final[-1]
            score_prev = float(prev.get("n_seed_ids", 0) or 0) + 0.01 * float(prev.get("snr_rms", 0) or 0)
            score_new = float(row.get("n_seed_ids", 0) or 0) + 0.01 * float(row.get("snr_rms", 0) or 0)
            if score_new > score_prev:
                final[-1] = row
        else:
            final.append(row)
    return pd.DataFrame(final).drop(columns=["_on"], errors="ignore").reset_index(drop=True)

## 6. Full-gather extraction, plotting, SEG-Y export, and picking

SEG-Y and plot geometry are based directly on position-coded station names, not trace index.

In [6]:
def read_full_gather_from_sds(label: str, event_time: UTCDateTime, stations: list[str]) -> Stream:
    network, location = parse_label(label)
    t1 = event_time - GATHER_PRE_S
    t2 = event_time + GATHER_POST_S
    st = read_deployment_from_sds(
        SDS_ROOT,
        network=network,
        location=location,
        station="*",
        channel="DP?",
        starttime=t1,
        endtime=t2,
        merge=True,
        verbose=False,
    )
    keep = set(stations)
    st = Stream([tr for tr in st if str(tr.stats.station) in keep and tr.stats.channel in EXTRACTION_CHANNELS])
    st.sort(keys=["station", "channel"])
    return st


def stream_to_component_arrays(st: Stream, component: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, list]:
    """Return time, data(ntr,npts), receiver_x_m, ordered traces for one component."""
    component = component.upper()
    traces = [tr for tr in st if channel_to_component(tr.stats.channel) == component]
    traces = sorted(traces, key=lambda tr: (station_to_x_m(tr.stats.station), tr.stats.station))
    if not traces:
        return np.array([]), np.empty((0, 0)), np.array([]), []
    npts = min(tr.stats.npts for tr in traces)
    dt = float(traces[0].stats.delta)
    time = np.arange(npts) * dt
    data = np.vstack([np.asarray(tr.data[:npts], dtype=np.float32) for tr in traces])
    rx = np.asarray([station_to_x_m(tr.stats.station) for tr in traces], dtype=float)
    return time, data, rx, traces


def write_component_products(
    st_event: Stream,
    label: str,
    event_id: str,
    gather_id: str,
    source_x_m: float | None,
) -> tuple[list[dict], list[dict]]:
    """Write MiniSEED, component SEG-Y and PNG files. Return file rows and trace-index rows."""
    network, location = parse_label(label)
    event_dir = OUT_ROOT / label
    mseed_dir = event_dir / "gathers_mseed"
    segy_dir = event_dir / "gathers_segy"
    fig_dir = event_dir / "figures"
    for d in [mseed_dir, segy_dir, fig_dir]:
        d.mkdir(parents=True, exist_ok=True)

    file_rows = []
    trace_rows = []
    source_x_for_headers = 0.0 if source_x_m is None or not np.isfinite(source_x_m) else float(source_x_m)

    # Full 3C MiniSEED
    mseed_path = mseed_dir / f"{event_id}_DPall.mseed"
    if WRITE_MSEED:
        st_event.write(str(mseed_path), format="MSEED")
        file_rows.append({
            "gather_file_id": f"GF_{event_id}_MSEED_3C",
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "line": network, "network": network, "location": location, "timewindow_label": label,
            "instrument_system": "nodal", "component": "3C", "file_type": "mseed",
            "file_path": str(mseed_path), "n_traces": len(st_event),
            "n_receivers": len(stream_stations(st_event)),
            "sample_rate_hz": float(st_event[0].stats.sampling_rate) if len(st_event) else np.nan,
            "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
            "processing_level": "raw_fullnode",
        })

    # Trace index points to the MiniSEED bundle by default.
    for i, tr in enumerate(sorted(st_event, key=lambda tr: (station_to_x_m(tr.stats.station), tr.stats.channel))):
        rx = station_to_x_m(tr.stats.station)
        sx = np.nan if source_x_m is None else source_x_m
        trace_rows.append({
            "trace_index_id": f"TR_{event_id}_{i:05d}",
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "line": network,
            "instrument_system": "nodal", "network": tr.stats.network,
            "station": tr.stats.station, "location": tr.stats.location,
            "channel": tr.stats.channel, "component": channel_to_component(tr.stats.channel),
            "receiver_x_m": rx, "source_x_m": sx,
            "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
            "starttime_utc": str(tr.stats.starttime),
            "sampling_rate_hz": float(tr.stats.sampling_rate), "npts": int(tr.stats.npts),
            "file_path": str(mseed_path), "trace_index": i,
            "amplitude_scale": 1.0,
        })

    # Component SEG-Y and figures
    for comp in ["Z", "N", "E"]:
        time, data, rx, traces = stream_to_component_arrays(st_event, comp)
        if len(traces) == 0:
            continue
        dt = float(traces[0].stats.delta)
        sr = float(traces[0].stats.sampling_rate)

        if WRITE_SEGY and HAVE_SEGY_TOOLS:
            segy_path = segy_dir / comp / f"{event_id}_{comp}.sgy"
            segy_path.parent.mkdir(parents=True, exist_ok=True)
            segy_st = gather_arrays_to_stream(
                data=data,
                dt_s=dt,
                starttime=traces[0].stats.starttime,
                receiver_x_m=rx,
                source_x_m=source_x_for_headers,
                shot_number=int(event_id.split("E")[-1]) if "E" in event_id else 1,
                station_prefix="R",
                network=network,
                component=comp,
            )
            write_segy(segy_st, segy_path)
            file_rows.append({
                "gather_file_id": f"GF_{event_id}_{comp}_SEGY",
                "run_id": RUN_ID, "event_id": event_id, "gather_id": f"{gather_id}_{comp}",
                "line": network, "network": network, "location": location, "timewindow_label": label,
                "instrument_system": "nodal", "component": comp, "file_type": "segy",
                "file_path": str(segy_path), "n_traces": int(data.shape[0]),
                "n_receivers": int(len(rx)), "sample_rate_hz": sr,
                "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
                "processing_level": "raw_fullnode_component",
            })

        if WRITE_PNG and HAVE_SEGY_TOOLS:
            wiggle_path = fig_dir / f"wiggle_{comp}" / f"{event_id}_{comp}_wiggle.png"
            image_path = fig_dir / f"image_{comp}" / f"{event_id}_{comp}_image.png"
            title = f"{event_id} {comp} — {label}"
            plot_wiggle_gather(
                time, data, rx,
                source_x_m=source_x_m,
                title=title,
                tmin=0.0, tmax=min(GATHER_PRE_S + GATHER_POST_S, 1.0),
                scale=0.8, clip_percentile=99, normalize=True,
                outfile=wiggle_path,
            )
            plot_image_gather(
                time, data, rx,
                source_x_m=source_x_m,
                title=title,
                tmin=0.0, tmax=min(GATHER_PRE_S + GATHER_POST_S, 1.0),
                clip_percentile=98,
                outfile=image_path,
            )
            for path, ftype in [(wiggle_path, "png_wiggle"), (image_path, "png_image")]:
                file_rows.append({
                    "gather_file_id": f"GF_{event_id}_{comp}_{ftype.upper()}",
                    "run_id": RUN_ID, "event_id": event_id, "gather_id": f"{gather_id}_{comp}",
                    "line": network, "network": network, "location": location, "timewindow_label": label,
                    "instrument_system": "nodal", "component": comp, "file_type": ftype,
                    "file_path": str(path), "n_traces": int(data.shape[0]),
                    "n_receivers": int(len(rx)), "sample_rate_hz": sr,
                    "pre_s": GATHER_PRE_S, "post_s": GATHER_POST_S,
                    "processing_level": "raw_fullnode_component",
                })

    return file_rows, trace_rows


def pick_event_stream(st_event: Stream, event_id: str, gather_id: str, source_x_m: float | None) -> pd.DataFrame:
    """Run GeoPark-style component pickers and station consensus on one full shot gather."""
    rows = []
    if len(st_event) == 0:
        return pd.DataFrame()

    st_pick = preprocess_for_picking(st_event.copy(), pick_cfg)
    event_start = min(tr.stats.starttime for tr in st_pick)

    for station in stream_stations(st_pick):
        st_sta = st_pick.select(station=station)
        if len(st_sta) == 0:
            continue
        picks_by_comp = {}
        tr_z = tr_n = tr_e = None
        for comp in ["Z", "N", "E"]:
            sel = [tr for tr in st_sta if channel_to_component(tr.stats.channel) == comp]
            if not sel:
                continue
            tr = sel[0]
            if comp == "Z": tr_z = tr
            if comp == "N": tr_n = tr
            if comp == "E": tr_e = tr
            try:
                picks_by_comp[comp] = pick_baer_aic_on_trace(tr)
            except Exception as e:
                picks_by_comp[comp] = {"aic_ok": False, "baer_ok": False, "aic_error": str(e), "baer_error": str(e)}

        p_ar = s_ar = None
        if pick_cfg.include_ar_s and tr_z is not None and tr_n is not None and tr_e is not None:
            try:
                p_ar, s_ar = try_ar_pick_short_station(tr_z, tr_n, tr_e, f1=pick_cfg.freqmin, f2=pick_cfg.freqmax)
            except Exception:
                pass

        if tr_z is not None and picks_by_comp:
            consensus = consensus_pick_for_station(
                picks_by_comp,
                tr_z,
                p_ar=p_ar,
                s_ar=s_ar,
                pick_tolerance_s=pick_cfg.pick_tolerance_s,
                min_votes=pick_cfg.min_votes,
                min_weight=pick_cfg.min_weight,
                include_ar_s=pick_cfg.include_ar_s,
                baer_weight=pick_cfg.baer_weight,
                mute_seconds=pick_cfg.mute_seconds,
            )
        else:
            consensus = {"ok": False, "time": None, "relative_s": np.nan, "n_votes": 0, "weight": 0, "methods": [], "components": []}

        rx = station_to_x_m(station)
        sx = np.nan if source_x_m is None else source_x_m

        # Individual picker rows
        for comp, d in picks_by_comp.items():
            tr = [tr for tr in st_sta if channel_to_component(tr.stats.channel) == comp][0]
            for method in ["aic", "baer"]:
                ok = bool(d.get(f"{method}_ok", False))
                t = d.get(f"{method}_time") if ok else None
                rows.append({
                    "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
                    "instrument_system": "nodal", "network": tr.stats.network,
                    "station": station, "location": tr.stats.location,
                    "channel": tr.stats.channel, "component": comp,
                    "receiver_x_m": rx, "source_x_m": sx,
                    "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
                    "pick_time_utc": str(t) if t is not None else None,
                    "pick_time_relative_s": float(t - event_start) if t is not None else np.nan,
                    "phase": "P?", "picker": method,
                    "pick_quality": "candidate" if ok else "failed",
                    "snr": np.nan,
                    "details_json": json.dumps({k: str(v) for k, v in d.items()}),
                })

        # Consensus row
        if consensus.get("ok"):
            t = consensus.get("time")
            pick_time_utc = str(t)
            rel = float(t - event_start)
            quality = "consensus"
        else:
            pick_time_utc = None
            rel = np.nan
            quality = "failed"
        rows.append({
            "run_id": RUN_ID, "event_id": event_id, "gather_id": gather_id,
            "instrument_system": "nodal", "network": st_sta[0].stats.network,
            "station": station, "location": st_sta[0].stats.location,
            "channel": "DP?", "component": "3C",
            "receiver_x_m": rx, "source_x_m": sx,
            "offset_m": np.nan if not np.isfinite(sx) else rx - sx,
            "pick_time_utc": pick_time_utc,
            "pick_time_relative_s": rel,
            "phase": "P?", "picker": "consensus",
            "pick_quality": quality,
            "snr": np.nan,
            "details_json": json.dumps({
                "votes": consensus.get("n_votes"),
                "weight": consensus.get("weight"),
                "methods": consensus.get("methods"),
                "components": consensus.get("components"),
                "p_ar_s": p_ar,
                "s_ar_s": s_ar,
            }),
        })
    return pd.DataFrame(rows)

## 7. Optional metadata matching hook

This is intentionally conservative. If no reliable metadata match is found, the event is still written with `metadata_match_status='unmatched_detection'` and `source_x_m=NULL`.

You can later improve this function using Jochen's spreadsheet columns once the shot-time table is finalized.

In [7]:

# Load Geode/field metadata from the current Excel workbook when available.
# This makes the notebook robust to renamed surveys such as:
#   - "Streamer/MASW main transect" -> T1_Streamer_MASW
#   - "Streamer/MASW western transect" -> T1A_Streamer_MASW
# and to T1 streamer rows whose `transect` values are shot labels (T1, T2, T3, ...),
# not actual transect names.
#
# Matching here is only approximate annotation. The authoritative source-position
# matching and stack-building should be handled later by notebooks 95/96.

def normalize_survey_name(sheet: str | None, survey: str | None = None) -> str:
    s = str(survey).strip() if survey is not None and str(survey) != "nan" else ""
    sh = str(sheet).strip() if sheet is not None else ""
    if sh in SHEET_TO_SURVEY:
        return SHEET_TO_SURVEY[sh]
    low = s.lower()
    if "main transect" in low or ("streamer" in low and "main" in low):
        return "T1_Streamer_MASW"
    if "western" in low or "t1a" in low:
        return "T1A_Streamer_MASW"
    if "1m" in low or "1 m" in low:
        if "t3" in low:
            return "T3_1m_refraction"
        if "t4" in low:
            return "T4_1m_refraction"
        return "T1_1m_refraction"
    if "2m" in low or "2 m" in low:
        return "T1_2m_refraction"
    return s or sh


def canonical_line_from_sheet(sheet: str, row: pd.Series) -> str:
    # Important: in T1_Streamer_MASW, the spreadsheet's `transect` column contains
    # values like T1, T2, ..., T83 that are shot labels, not line names. Use the
    # sheet name as source of truth for line/transect.
    if sheet == "T1_Streamer_MASW":
        return "T1"
    if sheet == "T1A_Streamer_MASW":
        return "T1A"
    if sheet.startswith("T1_"):
        return "T1"
    if sheet.startswith("T3_"):
        return "T3"
    if sheet.startswith("T4_"):
        return "T4"
    tr = str(row.get("transect", "")).strip()
    return tr if tr and tr != "nan" else ""


def source_x_from_metadata_row(row: pd.Series):
    for col in ["source_position_m", "shot_location_m"]:
        if col in row.index:
            x = pd.to_numeric(pd.Series([row.get(col)]), errors="coerce").iloc[0]
            if np.isfinite(x):
                return float(x)
    return np.nan


def corrected_geode_final_time(row: pd.Series, survey_name: str):
    if "geode_laptop_starttime" not in row.index:
        return pd.NaT
    t = pd.to_datetime(row.get("geode_laptop_starttime"), errors="coerce", utc=True)
    if pd.isna(t):
        return pd.NaT
    correction_s = SURVEY_TIME_MODELS.get(survey_name, {}).get("correction_s", 0.0)
    return t + pd.to_timedelta(correction_s, unit="s")


def load_metadata_events_from_workbook(workbook: Path | None) -> pd.DataFrame:
    if workbook is None or not Path(workbook).exists():
        return pd.DataFrame()

    sheets = [
        "T1_1m_Refraction",
        "T1_2m_Refraction",
        "T1_Streamer_MASW",
        "T1A_Streamer_MASW",
        "T3_1m_Refraction",
        "T4_1m_Refraction",
    ]
    rows = []
    for sheet in sheets:
        try:
            df = pd.read_excel(workbook, sheet_name=sheet)
        except Exception:
            continue
        for _, r in df.iterrows():
            survey_name = normalize_survey_name(sheet, r.get("survey"))
            line = canonical_line_from_sheet(sheet, r)
            x = source_x_from_metadata_row(r)
            final_time = corrected_geode_final_time(r, survey_name)
            duration_s = stack_duration_from_row(r)
            if pd.notna(final_time):
                stack_start = final_time - pd.to_timedelta(duration_s, unit="s")
                stack_end = final_time + pd.to_timedelta(FINAL_TRIGGER_MARGIN_S, unit="s")
            else:
                stack_start = pd.NaT
                stack_end = pd.NaT

            file_no = pd.to_numeric(pd.Series([r.get("file_no")]), errors="coerce").iloc[0] if "file_no" in r.index else np.nan
            shot_no = pd.to_numeric(pd.Series([r.get("shot_no")]), errors="coerce").iloc[0] if "shot_no" in r.index else np.nan
            event_id = f"GEODE_{sheet.upper()}_F{int(file_no) if np.isfinite(file_no) else len(rows)+1}"

            rows.append({
                "event_id": event_id,
                "instrument_system": "geode",
                "line": line,
                "transect": line,
                "survey": survey_name,
                "survey_type": "geode_stack_metadata",
                "shot_no": None if not np.isfinite(shot_no) else int(shot_no),
                "file_no": None if not np.isfinite(file_no) else int(file_no),
                "source_x_m": x,
                "source_type": r.get("source_type", "hammer"),
                "n_blows": r.get("n_blows", None),
                "n_shots": r.get("n_shots", None),
                "operator": r.get("operator", None),
                "plate_type": r.get("plate_type", None),
                "geode_laptop_starttime": r.get("geode_laptop_starttime", None),
                "geode_final_trigger_utc_dt": final_time,
                "geode_stack_start_dt": stack_start,
                "geode_stack_end_dt": stack_end,
                "stack_duration_s": duration_s,
                "metadata_source_sheet": sheet,
                "comment": r.get("comment", r.get("comments", None)),
                "confidence": r.get("confidence", None),
                "review_status": r.get("review_status", None),
            })
    out = pd.DataFrame(rows)
    if len(out):
        out["line"] = out["line"].astype(str)
        out["source_x_m"] = pd.to_numeric(out["source_x_m"], errors="coerce")
    return out


def load_indexed_metadata_events(conn: sqlite3.Connection) -> pd.DataFrame:
    """Fallback: load any Geode metadata rows already stored in SQLite."""
    try:
        df = pd.read_sql(
            """
            SELECT event_id, instrument_system, line, transect, survey, survey_type,
                   shot_no, file_no, source_x_m, source_type, n_blows, n_shots,
                   operator, plate_type, shot_time_utc, metadata_source_sheet,
                   comment, confidence, review_status
            FROM shot_events
            WHERE instrument_system = 'geode'
              AND source_x_m IS NOT NULL
            """,
            conn,
        )
    except Exception as e:
        print("Could not load indexed Geode metadata:", e)
        return pd.DataFrame()

    if len(df):
        df["survey"] = [normalize_survey_name(sh, sv) for sh, sv in zip(df.get("metadata_source_sheet", ""), df.get("survey", ""))]
        df["shot_time_dt"] = pd.to_datetime(df.get("shot_time_utc"), utc=True, errors="coerce")
        df["geode_final_trigger_utc_dt"] = df["shot_time_dt"]
        df["geode_stack_start_dt"] = df["shot_time_dt"] - pd.to_timedelta(DEFAULT_STACK_DURATION_S, unit="s")
        df["geode_stack_end_dt"] = df["shot_time_dt"] + pd.to_timedelta(FINAL_TRIGGER_MARGIN_S, unit="s")
        df["line"] = df["line"].astype(str)
    return df


metadata_events = load_metadata_events_from_workbook(METADATA_WORKBOOK)
if len(metadata_events) == 0:
    metadata_events = load_indexed_metadata_events(conn)

print("metadata events available for approximate stack-window matching:", len(metadata_events))
if len(metadata_events):
    display(metadata_events.groupby(["line", "survey", "metadata_source_sheet"]).size().reset_index(name="n"))
    display(metadata_events[["event_id", "line", "survey", "file_no", "source_x_m", "geode_final_trigger_utc_dt", "geode_stack_start_dt", "geode_stack_end_dt"]].head(10))


def _empty_meta_status(status: str, dt=None):
    return {
        "source_x_m": None,
        "source_type": None,
        "operator": None,
        "plate_type": None,
        "n_blows": None,
        "n_shots": None,
        "matched_metadata_event_id": None,
        "metadata_match_status": status,
        "metadata_match_time_error_s": dt,
        "survey": None,
        "survey_type": None,
        "shot_no": None,
        "file_no": None,
        "notes": None,
    }


def match_metadata_for_detection(label: str, detection_time: UTCDateTime) -> dict:
    """Approximate annotation of a nodal detection using Geode stack windows.

    This treats Geode file time as the final trigger in a stack and applies the
    survey-specific clock correction above. It does NOT use source-position
    estimates, so later notebooks should refine these associations.
    """
    network, location = parse_label(label)
    if len(metadata_events) == 0:
        return _empty_meta_status("unmatched_no_metadata_events")

    det_dt = pd.to_datetime(str(detection_time), utc=True)
    candidates = metadata_events[metadata_events["line"].astype(str) == str(network)].copy()
    candidates = candidates[candidates["geode_stack_start_dt"].notna() & candidates["geode_stack_end_dt"].notna()].copy()

    if len(candidates) == 0:
        return _empty_meta_status("unmatched_no_same_line_timed_metadata")

    inside = candidates[(candidates["geode_stack_start_dt"] <= det_dt) & (det_dt <= candidates["geode_stack_end_dt"])].copy()
    if len(inside) == 0:
        candidates["dt_abs_s"] = (candidates["geode_final_trigger_utc_dt"] - det_dt).dt.total_seconds().abs()
        best_dt = float(candidates["dt_abs_s"].min()) if len(candidates) else None
        return _empty_meta_status(f"unmatched_no_stack_window_contains_detection_nearest_{best_dt:.3f}s", best_dt)

    # Prefer the Geode stack whose final trigger is closest after/near the nodal event.
    inside["dt_to_final_s"] = (inside["geode_final_trigger_utc_dt"] - det_dt).dt.total_seconds()
    inside["dt_abs_s"] = inside["dt_to_final_s"].abs()
    best = inside.sort_values(["dt_abs_s", "file_no"]).iloc[0]

    def _int_or_none(x):
        try:
            return None if pd.isna(x) else int(float(x))
        except Exception:
            return None

    return {
        "source_x_m": float(best["source_x_m"]) if np.isfinite(best.get("source_x_m", np.nan)) else None,
        "source_type": best.get("source_type"),
        "operator": best.get("operator"),
        "plate_type": best.get("plate_type"),
        "n_blows": _int_or_none(best.get("n_blows")),
        "n_shots": _int_or_none(best.get("n_shots")),
        "matched_metadata_event_id": best.get("event_id"),
        "metadata_match_status": "matched_geode_stack_window_time_only",
        "metadata_match_time_error_s": float(best.get("dt_to_final_s")),
        "survey": best.get("survey"),
        "survey_type": best.get("survey_type"),
        "shot_no": _int_or_none(best.get("shot_no")),
        "file_no": _int_or_none(best.get("file_no")),
        "notes": best.get("comment"),
    }


metadata events available for approximate stack-window matching: 327


,line,survey,metadata_source_sheet,n
0,T1,T1_1m_refraction,T1_1m_Refraction,46
1,T1,T1_2m_refraction,T1_2m_Refraction,42
2,T1,T1_Streamer_MASW,T1_Streamer_MASW,83
3,T1A,T1A_Streamer_MASW,T1A_Streamer_MASW,87
4,T3,T3_1m_refraction,T3_1m_Refraction,39
5,T4,T4_1m_refraction,T4_1m_Refraction,30


,event_id,line,survey,file_no,source_x_m,geode_final_trigger_utc_dt,geode_stack_start_dt,geode_stack_end_dt
0,GEODE_T1_1M_REFRACTION_F3001,T1,T1_1m_refraction,3001,82.5,NaT,NaT,NaT
1,GEODE_T1_1M_REFRACTION_F3002,T1,T1_1m_refraction,3002,82.5,NaT,NaT,NaT
2,GEODE_T1_1M_REFRACTION_F3003,T1,T1_1m_refraction,3003,82.5,NaT,NaT,NaT
3,GEODE_T1_1M_REFRACTION_F3004,T1,T1_1m_refraction,3004,82.5,NaT,NaT,NaT
4,GEODE_T1_1M_REFRACTION_F3005,T1,T1_1m_refraction,3005,82.5,2026-05-18 16:03:42.500000+00:00,2026-05-18 16:02:48.500000+00:00,2026-05-18 16:03:47.500000+00:00
5,GEODE_T1_1M_REFRACTION_F3006,T1,T1_1m_refraction,3006,84.5,2026-05-18 16:08:30.500000+00:00,2026-05-18 16:07:36.500000+00:00,2026-05-18 16:08:35.500000+00:00
6,GEODE_T1_1M_REFRACTION_F3007,T1,T1_1m_refraction,3007,86.5,2026-05-18 16:13:29.500000+00:00,2026-05-18 16:12:35.500000+00:00,2026-05-18 16:13:34.500000+00:00
7,GEODE_T1_1M_REFRACTION_F3008,T1,T1_1m_refraction,3008,88.5,2026-05-18 16:18:32.500000+00:00,2026-05-18 16:17:38.500000+00:00,2026-05-18 16:18:37.500000+00:00
8,GEODE_T1_1M_REFRACTION_F3009,T1,T1_1m_refraction,3009,90.5,2026-05-18 16:20:51.500000+00:00,2026-05-18 16:19:57.500000+00:00,2026-05-18 16:20:56.500000+00:00
9,GEODE_T1_1M_REFRACTION_F3010,T1,T1_1m_refraction,3010,92.5,2026-05-18 16:26:23.500000+00:00,2026-05-18 16:25:23.500000+00:00,2026-05-18 16:26:28.500000+00:00


## 8. Main processing loop

This is the expensive cell. Start with `RUN_LABELS = ['T1_N1_Streamer']` and `MAX_EVENTS_PER_WINDOW = 5` if testing.

In [8]:
labels_to_run = list(TIMEWINDOWS.keys()) if RUN_LABELS is None else list(RUN_LABELS)
all_window_catalogs = []

for label in labels_to_run:
    check_free_space(OUT_ROOT, MIN_FREE_GB)
    print("\n" + "="*100)
    print("Processing", label)
    start, end = TIMEWINDOWS[label]
    network, location = parse_label(label)
    geometry_id = f"{label}_{network}_{location}_DPall"

    label_out = OUT_ROOT / label
    (label_out / "tables").mkdir(parents=True, exist_ok=True)

    # 1. Discover all DPZ stations for this network/location after 86_downsample.
    stations = discover_sds_stations(SDS_ROOT, network, location, DETECTION_CHANNEL, EXCLUDE_STATIONS)
    print(f"Discovered {len(stations)} {network}.{location}.{DETECTION_CHANNEL} stations")
    print(stations)
    if len(stations) == 0:
        continue

    # 2. Write receiver geometry to SQLite.
    geom_df = build_geometry_df(label, start, end, stations)
    geom_df.to_sql("receiver_geometry", conn, if_exists="append", index=False)
    geom_df.to_csv(label_out / "tables" / f"{label}_receiver_geometry.csv", index=False)

    # 3. Run chunked DPZ detection.
    raw_det = run_detection_for_window(label, start, end, stations)
    if len(raw_det) == 0:
        print("No detections")
        continue
    raw_path = label_out / "tables" / f"{label}_detected_events_raw_chunked.csv"
    raw_det.to_csv(raw_path, index=False)

    merged_det = merge_detection_catalog(raw_det, tolerance_s=MERGE_TOLERANCE_S, min_spacing_s=MIN_EVENT_SPACING_S)
    merged_path = label_out / "tables" / f"{label}_detected_events_merged.csv"
    merged_det.to_csv(merged_path, index=False)
    print(f"Raw detections: {len(raw_det)}; merged detections: {len(merged_det)}")

    if MAX_EVENTS_PER_WINDOW is not None:
        merged_det = merged_det.iloc[:MAX_EVENTS_PER_WINDOW].copy()

    # 4. For each detection, go back to SDS and extract all DP[ENZ] stations.
    for local_i, row in merged_det.reset_index(drop=True).iterrows():
        event_num = local_i + 1
        event_id = f"{label}_{network}_{location}_E{event_num:05d}"
        gather_id = f"{event_id}_nodal_DPall"
        print(f"\n{event_id}: on_time={row['on_time']}")

        try:
            event_time = UTCDateTime(row["on_time"])
            meta = match_metadata_for_detection(label, event_time)
            source_x_m = meta.get("source_x_m")

            st_event = read_full_gather_from_sds(label, event_time, stations)
            n_receivers = len(stream_stations(st_event))
            n_traces = len(st_event)
            print(f"  extracted {n_traces} traces, {n_receivers} receivers")

            # If extraction fails, still catalog the detected event.
            status = "ok" if len(st_event) else "no_traces_extracted"

            event_row = {
                "run_id": RUN_ID,
                "event_id": event_id,
                "instrument_system": "nodal",
                "line": network,
                "network": network,
                "location": location,
                "timewindow_label": label,
                "geometry_id": geometry_id,
                "detection_time_utc": utc_to_str(row.get("on_time")),
                "on_time_utc": utc_to_str(row.get("on_time")),
                "off_time_utc": utc_to_str(row.get("off_time")),
                "duration_s": float(row.get("duration_s", np.nan)) if pd.notna(row.get("duration_s", np.nan)) else np.nan,
                "source_x_m": source_x_m,
                "source_type": meta.get("source_type"),
                "operator": meta.get("operator"),
                "plate_type": meta.get("plate_type"),
                "n_blows": meta.get("n_blows"),
                "n_shots": meta.get("n_shots"),
                "survey": meta.get("survey"),
                "survey_type": meta.get("survey_type") or "nodal_detection",
                "shot_no": meta.get("shot_no"),
                "file_no": meta.get("file_no"),
                "matched_metadata_event_id": meta.get("matched_metadata_event_id"),
                "metadata_match_status": meta.get("metadata_match_status", "unmatched_detection"),
                "metadata_match_time_error_s": meta.get("metadata_match_time_error_s"),
                "detection_n_seed_ids": int(row.get("n_seed_ids", 0) or 0),
                "detection_n_stations": int(row.get("n_stations", 0) or 0) if "n_stations" in row else None,
                "detection_seed_ids": str(row.get("seed_ids", "")),
                "detection_stations": str(row.get("stations", "")),
                "snr_rms": float(row.get("snr_rms", np.nan)) if pd.notna(row.get("snr_rms", np.nan)) else np.nan,
                "n_receivers_extracted": n_receivers,
                "n_traces_extracted": n_traces,
                "status": status,
                "notes": meta.get("notes"),
            }
            pd.DataFrame([event_row]).to_sql("shot_events", conn, if_exists="append", index=False)

            if len(st_event) == 0:
                conn.commit()
                continue

            file_rows, trace_rows = write_component_products(st_event, label, event_id, gather_id, source_x_m)
            if file_rows:
                pd.DataFrame(file_rows).to_sql("shot_gather_files", conn, if_exists="append", index=False)
            if trace_rows:
                pd.DataFrame(trace_rows).to_sql("trace_index", conn, if_exists="append", index=False)

            picks_df = pick_event_stream(st_event, event_id, gather_id, source_x_m)
            if len(picks_df):
                pd.DataFrame(picks_df).to_sql("picks", conn, if_exists="append", index=False)
                picks_out = label_out / "tables" / "picks"
                picks_out.mkdir(parents=True, exist_ok=True)
                picks_df.to_csv(picks_out / f"{event_id}_picks.csv", index=False)

            conn.commit()
        except Exception as e:
            print("  FAILED:", e)
            log_error(conn, "event_processing", label=label, event_id=event_id, exc=e)
            continue

print("Done. Catalog:", CATALOG_DB)


Processing T1_N1_Streamer
Discovered 35 T1.N1.DPZ stations
['02800', '03600', '04400', '05207', '06004', '06809', '07600', '08003', '08405', '08808', '09210', '09605', '10000', '10398', '10800', '11202', '11600', '12005', '12402', '12801', '13202', '13605', '14000', '14400', '14800', '15195', '16000', '16400', '16809', '17604', '18405', '19200', '20000', '20800', '21600']
T1_N1_Streamer chunk 0: 2026-05-16T17:13:00.000000Z to 2026-05-16T17:14:00.000000Z
  read 15 traces, 15 stations
  detections: 8
T1_N1_Streamer chunk 1: 2026-05-16T17:14:00.000000Z to 2026-05-16T17:15:00.000000Z
  read 15 traces, 15 stations
  detections: 3
T1_N1_Streamer chunk 2: 2026-05-16T17:15:00.000000Z to 2026-05-16T17:16:00.000000Z
  read 15 traces, 15 stations
  detections: 2
T1_N1_Streamer chunk 3: 2026-05-16T17:16:00.000000Z to 2026-05-16T17:17:00.000000Z
  read 15 traces, 15 stations
  detections: 4
T1_N1_Streamer chunk 4: 2026-05-16T17:17:00.000000Z to 2026-05-16T17:18:00.000000Z
  read 15 traces, 15 stat

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:824: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


  extracted 45 traces, 15 receivers

T1_N1_Streamer_T1_N1_E00003: on_time=2026-05-16 17:13:28.030000


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:824: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


  extracted 45 traces, 15 receivers

T1_N1_Streamer_T1_N1_E00004: on_time=2026-05-16 17:13:30.858000


...

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:824: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


  extracted 105 traces, 35 receivers

T3_N4_Refraction1am_T3_N4_E00523: on_time=2026-05-19 18:22:04.542000


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:824: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


  extracted 105 traces, 35 receivers
Done. Catalog: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:824: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


## 9. Catalog inspection

Use SQL queries to inspect what was produced.

In [9]:
def q(sql: str, params=None):
    return pd.read_sql(sql, conn, params=params or {})

print("processing_runs")
display(q("SELECT run_id, notebook_name, run_time_utc, input_sds_root, output_root FROM processing_runs ORDER BY run_time_utc DESC LIMIT 5"))

print("shot_events summary")
display(q("""
SELECT timewindow_label, status, COUNT(*) AS n_events,
       AVG(n_receivers_extracted) AS avg_receivers,
       MIN(n_receivers_extracted) AS min_receivers,
       MAX(n_receivers_extracted) AS max_receivers
FROM shot_events
WHERE run_id = :run_id
GROUP BY timewindow_label, status
ORDER BY timewindow_label, status
""", {"run_id": RUN_ID}))

print("files summary")
display(q("""
SELECT timewindow_label, file_type, component, COUNT(*) AS n_files
FROM shot_gather_files
WHERE run_id = :run_id
GROUP BY timewindow_label, file_type, component
ORDER BY timewindow_label, file_type, component
""", {"run_id": RUN_ID}))

print("pick summary")
display(q("""
SELECT timewindow_label, picker, pick_quality, COUNT(*) AS n
FROM picks p
JOIN shot_events e USING(event_id)
WHERE p.run_id = :run_id
GROUP BY timewindow_label, picker, pick_quality
ORDER BY timewindow_label, picker, pick_quality
""", {"run_id": RUN_ID}))

processing_runs


,run_id,notebook_name,run_time_utc,input_sds_root,output_root
0,20260617T230552Z_da97ed5d,90_nodal_fullnode_detection_pick_and_shotgathe...,2026-06-17T23:05:52.363740+00:00,/Volumes/tachyon/LBSSP_DATA/nodal_sds_position...,/Volumes/tachyon/LBSSP_DATA/nodal_fullnode_sho...


shot_events summary


,timewindow_label,status,n_events,avg_receivers,min_receivers,max_receivers
0,T1_N1_Streamer,ok,532,30.308271,15,35
1,T1_N2_Nodal1,ok,1376,32.166424,26,35
2,T1_N2_Nodal2,ok,81,34.000000,34,34
3,T1_N2_Refraction1m,ok,431,34.000000,34,34
4,T1_N2_Refraction2m,ok,450,34.000000,34,34
5,T1_N3_Nodal3,ok,43,34.000000,34,34
6,T3_N4_Refraction1am,ok,523,35.000000,35,35


files summary


,timewindow_label,file_type,component,n_files
0,T1_N1_Streamer,mseed,3C,532
1,T1_N2_Nodal1,mseed,3C,1376
2,T1_N2_Nodal2,mseed,3C,81
3,T1_N2_Refraction1m,mseed,3C,431
4,T1_N2_Refraction2m,mseed,3C,450
5,T1_N3_Nodal3,mseed,3C,43
6,T3_N4_Refraction1am,mseed,3C,523


pick summary


,timewindow_label,picker,pick_quality,n
0,T1_N1_Streamer,aic,candidate,48372
1,T1_N1_Streamer,baer,candidate,48372
2,T1_N1_Streamer,consensus,consensus,12755
3,T1_N1_Streamer,consensus,failed,3369
4,T1_N2_Nodal1,aic,candidate,132783
5,T1_N2_Nodal1,baer,candidate,132783
6,T1_N2_Nodal1,consensus,consensus,28727
7,T1_N2_Nodal1,consensus,failed,15534
8,T1_N2_Nodal2,aic,candidate,8262
9,T1_N2_Nodal2,baer,candidate,8262


## 10. Optional CSV exports

SQLite is the source of truth, but these CSV snapshots are convenient for inspection, sharing, and debugging.

In [10]:
if EXPORT_CSV_SNAPSHOTS:
    tables = [
        "processing_runs",
        "receiver_geometry",
        "shot_events",
        "shot_gather_files",
        "trace_index",
        "picks",
        "processing_errors",
    ]
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table} WHERE run_id = ?" if table != "processing_runs" else "SELECT * FROM processing_runs WHERE run_id = ?", conn, params=(RUN_ID,))
        out = CSV_EXPORT_DIR / f"{RUN_ID}_{table}.csv"
        df.to_csv(out, index=False)
        print(table, len(df), "->", out)

processing_runs 1 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_processing_runs.csv
receiver_geometry 245 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_receiver_geometry.csv
shot_events 3436 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_shot_events.csv
shot_gather_files 3436 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_shot_gather_files.csv
trace_index 338580 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_trace_index.csv
picks 790020 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_picks.csv
processing_errors 0 -> /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/catalog_exports/20260617T230552Z_da97ed5d_processing_errors.csv


## 11. Next notebooks

Recommended follow-ons:

- `91_stack_nodal_repeated_shots_by_metadata.ipynb`
- `92_extract_source_waveforms_and_spectra_from_catalog.ipynb`
- `93_compare_geode_streamer_nodal_common_shots.ipynb`
- `94_build_combined_supergathers.ipynb`

These should query `lbssp_shot_catalog.sqlite`, not reconstruct filenames manually.

# Testing
What remains are some suggestions by ChatGPT to check what we now have in the database

In [13]:
shot_events = pd.read_sql("SELECT * FROM shot_events LIMIT 5", conn)

print(shot_events.columns.tolist())

['event_id', 'instrument_system', 'line', 'transect', 'survey', 'survey_type', 'shot_no', 'file_no', 'source_x_m', 'source_type', 'n_blows', 'n_shots', 'operator', 'plate_type', 'shot_time_local', 'shot_time_utc', 'receiver_first_m', 'receiver_last_m', 'receiver_spacing_m', 'nominal_shot_spacing_m', 'geode_read_ok', 'geode_read_format', 'geode_n_traces', 'geode_sampling_rate_hz', 'geode_duration_s_first_trace', 'geode_file_path', 'geode_folder', 'geode_match_status', 'geode_match_note', 'source_page', 'confidence', 'review_status', 'comment', 'metadata_source_sheet', 'extra_json', 'run_id', 'network', 'location', 'timewindow_label', 'geometry_id', 'detection_time_utc', 'on_time_utc', 'off_time_utc', 'duration_s', 'matched_metadata_event_id', 'metadata_match_status', 'metadata_match_time_error_s', 'detection_n_seed_ids', 'detection_n_stations', 'detection_seed_ids', 'detection_stations', 'snr_rms', 'n_receivers_extracted', 'n_traces_extracted', 'status', 'notes']


In [14]:
import pandas as pd

shot_events = pd.read_sql(
    """
    SELECT
        timewindow_label,
        detection_time_utc,
        shot_time_utc,
        source_x_m,
        file_no,
        metadata_match_status
    FROM shot_events
    """,
    conn,
)

shot_events["detection_time_utc"] = pd.to_datetime(
    shot_events["detection_time_utc"],
    errors="coerce",
    utc=True,
)

shot_events["shot_time_utc"] = pd.to_datetime(
    shot_events["shot_time_utc"],
    errors="coerce",
    utc=True,
)

summary = (
    shot_events
    .groupby("timewindow_label")
    .agg(
        n_events=("detection_time_utc", "count"),
        first_detection=("detection_time_utc", "min"),
        last_detection=("detection_time_utc", "max"),
        n_geode_matches=("shot_time_utc", lambda x: x.notna().sum()),
    )
    .reset_index()
)

display(summary)

,timewindow_label,n_events,first_detection,last_detection,n_geode_matches
0,T1_N1_Streamer,532,2026-05-16 17:13:01.890000+00:00,2026-05-16 21:44:03.302000+00:00,0
1,T1_N2_Nodal1,1376,2026-05-17 16:19:34.132000+00:00,2026-05-17 18:59:59.388000+00:00,0
2,T1_N2_Nodal2,81,2026-05-19 13:07:40.072000+00:00,2026-05-19 13:15:26.874000+00:00,0
3,T1_N2_Refraction1m,431,2026-05-18 16:07:22.442000+00:00,2026-05-18 18:38:21.774000+00:00,0
4,T1_N2_Refraction2m,450,2026-05-18 20:17:03.646000+00:00,2026-05-18 23:13:08.346000+00:00,0
5,T1_N3_Nodal3,43,2026-05-19 14:00:18.624000+00:00,2026-05-19 14:03:18.740000+00:00,0
6,T3_N4_Refraction1am,523,2026-05-19 16:02:33.030000+00:00,2026-05-19 18:22:04.542000+00:00,0


In [17]:
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn,
)


,name
0,picks
1,processing_errors
2,processing_runs
3,receiver_geometry
4,shot_events
5,shot_gather_files
6,trace_index
